# BIOT 6900 — Module 1 Starter Notebook
### Computational Environment Setup & Tool Ecosystem

**Name:** Shreegowri Mattighatta Bhojanna 
**Date:** 09/25/2026  

Work through this notebook top to bottom. Fill in every cell marked **TODO**, then commit the notebook to your `biot6900` GitHub repository and post the link on Canvas.

## How to use this notebook

- Run each cell in order with **Shift + Enter**.
- Cells marked **TODO** need something from you — an email, a dataset path, or a one-line note.
- If a cell errors, read the message, check the **Troubleshooting** section of the Lab Guide, or bring it to office hours.
- **Save often.** If you're on Colab, remember it forgets your files — commit to GitHub before you close the tab.
- **Platform:** Week 1 is designed to run **locally** in your `biot6900` conda environment. A Colab version is on Canvas if you need it.

## Part A — Environment verification
Confirm your environment is set up correctly before you go any further.

In [5]:
import sys, platform
print("Python version:   ", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Platform:         ", platform.platform())

Python version:    3.11.16
Python executable: C:\Users\mattighattabhojann.s\AppData\Local\anaconda3\envs\biot6900\python.exe
Platform:          Windows-10-10.0.26200-SP0


In [6]:
import os
# You should see "biot6900". If not, activate it in a terminal: conda activate biot6900
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "not detected"))

Conda environment: biot6900


In [22]:
# Git version (shell command). If this errors, install Git and restart the notebook.
'''
Since I am using northeastern loaner laptop 
- I do not have appropriate permissions to access the git from this laptop
- will use this version control tool once I set up my workstation and start using that new compute for the modules. 
'''
#!git --version

'\nSince I am using northeastern loaner laptop \n- I do not have appropriate permissions to access the git from this laptop\n- will use this version control tool once I set up my workstation and start using that new compute for the modules. \n'

In [8]:
import Bio, pandas as pd, numpy as np, requests
print("BioPython:", Bio.__version__)
print("pandas:   ", pd.__version__)
print("numpy:    ", np.__version__)
print("requests: ", requests.__version__)

BioPython: 1.88
pandas:    3.0.6
numpy:     2.4.6
requests:  2.34.2


**Checkpoint A:** all four package versions printed without error, and your environment shows `biot6900`. If anything failed, fix it before continuing.

## Part B — BioPython quickstart
A quick preview of the library you'll use all semester.

In [9]:
from Bio.Seq import Seq
dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")
print("Sequence length:   ", len(dna))
print("Protein translation:", dna.translate())

Sequence length:    24
Protein translation: MAIVMGR*


**TODO (B):** In one sentence, what did `.translate()` do to the DNA sequence?

*Translated DNA sequence (in bases form) to protein sequence (one letter amino acid sequence form).*

## Part C — Biomedical database API exploration
Run each query, then write your one-line note in the markdown cell that follows it. Each database uses the same rhythm: build a query, call the API, read the result.

### C1 — PubMed (via `Bio.Entrez`)

In [10]:
from Bio import Entrez
Entrez.email = "mattighattabhojann.s@northeastern.edu"
handle = Entrez.esearch(db="pubmed", term="ESMFold protein structure", retmax=10) #"AlphaFold protein structure"
record = Entrez.read(handle)
handle.close()
print("PubMed IDs:", record["IdList"])

PubMed IDs: ['42767986', '42762846', '42730146', '42711316', '42668358', '42628918', '42608825', '42560022', '42469326', '42467514']


**TODO (C1) — Note:** which search term did you use, and how many PMIDs came back?

*Used "ESMFold protein structure" search term and retrieved the 10 PMIDs related to that search term.*

### C2 — UniProt (REST API)

In [11]:
import requests
acc = "P07949" # #"P04637"   # human tumor-suppressor p53 ==> Cellular tumor antigen p53
r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json")
data = r.json()
name = data["proteinDescription"]["recommendedName"]["fullName"]["value"]
print("Accession:", acc)
print("Protein:  ", name)

Accession: P07949
Protein:   Proto-oncogene tyrosine-protein kinase receptor Ret


**TODO (C2) — Note:** which protein did you retrieve?

*Protein: Proto-oncogene tyrosine-protein kinase receptor Ret*

*Gene: RET - an oncogene*

### C3 — PDB (RCSB REST API)

In [12]:
import requests
pdb_id = "5V9O" #"1TUP"   # a p53 structure ==> TUMOR SUPPRESSOR P53 COMPLEXED WITH DNA
r = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
print("PDB ID:  ", pdb_id)
print("Title:   ", r.json()["struct"]["title"])
print("Coordinates:", f"https://files.rcsb.org/download/{pdb_id}.pdb")

PDB ID:   5V9O
Title:    KRAS G12C inhibitor
Coordinates: https://files.rcsb.org/download/5V9O.pdb


**TODO (C3) — Note:** what structure did you retrieve?

*KRAS G12C inhibitor.* 

### C4 — GWAS Catalog (EBI REST API)

In [13]:
import requests
base = "https://www.ebi.ac.uk/gwas/rest/api" #better than gnoMAD/clinvar?
rsid = "rs7412" ##   # a well-known APOE variant ==> missense_variant
r = requests.get(f"{base}/singleNucleotidePolymorphisms/{rsid}") #Only for SNPs? - did not work for "rs11061160" - 185delAG; "rs11061161";"rs121434569"
data = r.json()
print("Variant:        ", data.get("rsId"))
print("Functional class:", data.get("functionalClass"))

Variant:         rs7412
Functional class: missense_variant


**TODO (C4) — Note:** which variant did you look up, and one thing you noticed?

*The variant is missense meaning it changes the aminoacid coded in the resulting sequence.* 

*The requests.get focused on SNP search; since this did not work for other few of the mutation I was checking, I went with given rs number. Looks like rs number/dbSNP-ID varies between databases (and maybe specific for SNPs?).*

## Part D — Download & load sample multi-omics data
Accessions and download links are posted on Canvas. Download your assigned file(s), then load each one and confirm its dimensions.

**TODO (D) — Note:** for each dataset, record its shape and one sentence on what it contains.

_Your note:_ 

In [14]:
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------------
# Catch-up note:
# I missed the Week 1 class and therefore did not receive/record the originally assigned accession.
#
# To complete the data-loading exercise, I used the three CPTAC-BRCA teaching datasets available in the course materials.
# -------------------------------------------------------------------

data_dir = Path("..//data") #I have placed "data" folder at biot6900 since we will focus on particular dataset going forward

files = {
    "CPTAC-BRCA Mutation": data_dir / "cptac_brca_mutation.tsv",
    "CPTAC-BRCA RNA": data_dir / "cptac_brca_rna.tsv",
    "CPTAC-BRCA Protein": data_dir / "cptac_brca_protein.tsv",
}
datasets = {}
for name, path in files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    datasets[name] = df
    print("=" * 70)
    print(name)
    print("=" * 70)
    print("File:", path)
    print("Shape (rows, columns):", df.shape)
    print("Index name:", df.index.name)
    print("First 5 column names:", df.columns[:5].tolist())
    print("Missing values:", int(df.isna().sum().sum()))
    print()
    display(df.head())
    print()

CPTAC-BRCA Mutation
File: ..\data\cptac_brca_mutation.tsv
Shape (rows, columns): (800, 1)
Index name: None
First 5 column names: ['mut_freq']
Missing values: 0



,mut_freq
TP53,0.4130
PIK3CA,0.3691
ERBB2,0.0496
ESR1,0.0275
GATA3,0.4236



CPTAC-BRCA RNA
File: ..\data\cptac_brca_rna.tsv
Shape (rows, columns): (800, 100)
Index name: None
First 5 column names: ['BR000', 'BR001', 'BR002', 'BR003', 'BR004']
Missing values: 0



,BR000,BR001,BR002,BR003,BR004,BR005,BR006,BR007,BR008,BR009,...,BR090,BR091,BR092,BR093,BR094,BR095,BR096,BR097,BR098,BR099
TP53,2.9727,3.5514,3.9849,-0.1874,2.9027,2.4645,2.8135,2.4451,2.5374,2.0739,...,2.9045,2.5761,3.1964,3.2193,1.2993,1.1386,0.5888,3.5501,4.5066,1.3081
PIK3CA,0.4650,3.0012,2.3210,2.1630,0.8797,0.6642,1.9790,2.4266,1.5542,1.0873,...,1.4507,0.1844,2.6936,1.0591,1.7873,2.0204,1.4635,2.5599,2.1769,0.2067
ERBB2,1.8101,0.5920,1.4668,1.4670,0.1257,1.6350,1.8996,2.2596,0.8548,1.1865,...,1.5881,2.3698,1.2879,2.6245,2.7189,1.9047,1.6665,1.8276,0.7642,0.8728
ESR1,-0.5675,2.9497,1.8020,0.7518,3.5393,2.2821,-0.9975,2.3934,-0.4270,1.6048,...,3.3459,-0.5132,1.6969,1.6067,2.9049,2.0747,1.6497,0.8048,1.3454,1.6163
GATA3,3.0912,1.7799,1.8764,3.0029,1.4399,3.5722,1.5212,1.4808,2.0000,3.5519,...,3.3298,1.8130,2.9716,1.8212,2.5324,0.5998,2.7371,2.8900,1.7437,1.6981



CPTAC-BRCA Protein
File: ..\data\cptac_brca_protein.tsv
Shape (rows, columns): (500, 100)
Index name: None
First 5 column names: ['BR000', 'BR001', 'BR002', 'BR003', 'BR004']
Missing values: 0



,BR000,BR001,BR002,BR003,BR004,BR005,BR006,BR007,BR008,BR009,...,BR090,BR091,BR092,BR093,BR094,BR095,BR096,BR097,BR098,BR099
TP53,0.8155,2.1458,1.3357,-2.3153,1.3119,1.0408,1.7015,2.6375,1.6519,0.2022,...,2.3885,2.0881,1.5219,-0.2183,0.3105,-0.4739,-0.2452,2.9453,2.4921,-0.7555
PIK3CA,2.1710,0.3929,-2.9501,-1.0783,1.4258,0.3958,-0.5922,-0.0238,0.7585,0.6483,...,1.0588,0.4118,-0.0673,0.8747,-1.0801,1.3384,1.0585,0.1988,0.1490,0.2646
ERBB2,0.0028,-0.5573,1.9719,1.8423,1.0831,-0.0009,-0.3352,2.1333,-0.8336,0.7112,...,-1.3665,0.6416,1.1462,0.8443,0.4575,-0.6905,-0.6080,-0.6605,2.3108,0.2083
ESR1,1.2500,0.0631,-0.8207,-0.0640,0.8581,-0.2819,-0.4072,3.6670,0.7289,-0.2115,...,-0.1650,0.8216,1.1886,0.5481,2.3722,-1.4247,2.3472,-0.6542,1.7764,2.4747
GATA3,1.2538,2.3336,1.0145,1.6062,0.7040,1.7901,0.9052,0.6225,-0.4851,2.3387,...,1.7818,1.5317,-1.2014,1.0776,1.6768,-0.1209,0.7790,2.7439,1.7432,0.6907


In [15]:
'''
import pandas as pd
# TODO: set the path to your downloaded file (use sep="\t" for tab-separated files)
path = "data/TODO_assigned_file.csv" #cptac_brca_mutation.tsv; #cptac_brca_rna.tsv; #cptac_brca_protein.tsv
df = pd.read_csv(path)
print("Shape (rows, columns):", df.shape)
df.head()
'''

'\nimport pandas as pd\n# TODO: set the path to your downloaded file (use sep="\t" for tab-separated files)\npath = "data/TODO_assigned_file.csv" #cptac_brca_mutation.tsv; #cptac_brca_rna.tsv; #cptac_brca_protein.tsv\ndf = pd.read_csv(path)\nprint("Shape (rows, columns):", df.shape)\ndf.head()\n'

## Submission checklist

- [ ] Part A prints all versions and shows the `biot6900` environment
- [ ] Part C: all four queries ran, each with your one-line note
- [ ] Part D: dataset(s) loaded and shape printed
- [ ] Notebook committed to your `biot6900` GitHub repository
- [ ] Repository link posted on Canvas

_Graded 60 / 40: 60% clean execution, 40% interpretation & documentation._